In [2]:
import numpy as np
import pandas as pd
from scipy.stats import skew
from scipy.special import boxcox1p
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVR

In [3]:
DATA_DIR = "."
SEED = 42

In [4]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

In [5]:
# ---- 1. Remove the two well-known outliers (huge house, tiny price) ----
train = train[~((train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000))]

y = np.log1p(train["SalePrice"].values)  # competition metric = RMSE on log price
test_ids = test["Id"]
n_train = len(train)
df = pd.concat([train.drop(columns=["SalePrice", "Id"]),
                test.drop(columns=["Id"])], ignore_index=True)

In [6]:
# ---- 2. Missing values ----
# "NA" in these columns means the feature does not exist
none_cols = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
             "BsmtFinType2", "FireplaceQu", "GarageType", "GarageFinish",
             "GarageQual", "GarageCond", "PoolQC", "Fence", "MiscFeature",
             "MasVnrType"]
for c in none_cols:
    df[c] = df[c].fillna("None")

zero_cols = ["GarageYrBlt", "GarageArea", "GarageCars", "BsmtFinSF1", "BsmtFinSF2",
             "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"]
for c in zero_cols:
    df[c] = df[c].fillna(0)

In [7]:
# Lot frontage: median of the neighbourhood
df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
    lambda s: s.fillna(s.median()))
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

df["Functional"] = df["Functional"].fillna("Typ")
df = df.drop(columns=["Utilities"])  # almost constant
for c in ["MSZoning", "Electrical", "KitchenQual", "Exterior1st", "Exterior2nd",
          "SaleType"]:
    df[c] = df[c].fillna(df[c].mode()[0])


In [8]:
# ---- 3. Ordinal encoding of quality-type columns ----
qual = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
for c in ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
          "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]:
    df[c] = df[c].map(qual).fillna(0).astype(int)
df["BsmtExposure"] = df["BsmtExposure"].map(
    {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}).astype(int)
fin = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
df["BsmtFinType1"] = df["BsmtFinType1"].map(fin).astype(int)
df["BsmtFinType2"] = df["BsmtFinType2"].map(fin).astype(int)
df["GarageFinish"] = df["GarageFinish"].map({"None": 0, "Unf": 1, "RFn": 2, "Fin": 3}).astype(int)

In [9]:
# Numeric codes that are really categories
for c in ["MSSubClass", "MoSold", "YrSold"]:
    df[c] = df[c].astype(str)

In [10]:
# ---- 4. Feature engineering ----
df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
df["TotalBath"] = (df["FullBath"] + 0.5 * df["HalfBath"] +
                   df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"])
df["TotalPorchSF"] = (df["OpenPorchSF"] + df["EnclosedPorch"] + df["3SsnPorch"] +
                      df["ScreenPorch"] + df["WoodDeckSF"])
yr_sold = df["YrSold"].astype(int)
df["HouseAge"] = yr_sold - df["YearBuilt"]
df["RemodAge"] = yr_sold - df["YearRemodAdd"]
df["WasRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
df["IsNew"] = (yr_sold == df["YearBuilt"]).astype(int)
df["HasPool"] = (df["PoolArea"] > 0).astype(int)
df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)
df["QualSF"] = df["OverallQual"] * df["TotalSF"]
df["QualGrLiv"] = df["OverallQual"] * df["GrLivArea"]

In [11]:
# ---- 5. Fix skewed numeric features (Box-Cox) ----
num_cols = df.select_dtypes(include=[np.number]).columns
skews = df[num_cols].apply(lambda s: skew(s.dropna()))
for c in skews[abs(skews) > 0.75].index:
    if df[c].nunique() > 5:  # skip binary-like flags
        df[c] = boxcox1p(df[c].clip(lower=0), 0.15)

In [12]:
# ---- 6. One-hot encode and split back ----
df = pd.get_dummies(df, dtype=float)
X = df.iloc[:n_train].values
X_test = df.iloc[n_train:].values
print(f"Features: {X.shape[1]}  | train rows: {X.shape[0]}  | test rows: {X_test.shape[0]}")


Features: 284  | train rows: 1458  | test rows: 1459


In [13]:
# ---- 7. Models (tuned) ----
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import RidgeCV, LassoCV
import warnings; warnings.filterwarnings("ignore")

def rmse(m, X_, y_, cv):
    return np.sqrt(-cross_val_score(m, X_, y_, scoring="neg_mean_squared_error", cv=cv, n_jobs=-1)).mean()

kf = KFold(5, shuffle=True, random_state=SEED)

In [14]:
# quick tuning of the linear/kernel models by CV
best = {}
for a in [0.0002, 0.0003, 0.0004, 0.0005, 0.0007]:
    best[("lasso", a)] = rmse(make_pipeline(RobustScaler(), Lasso(alpha=a, max_iter=50000)), X, y, kf)
a_l = min([k for k in best if k[0]=="lasso"], key=best.get)[1]
print("best Lasso alpha", a_l, best[("lasso", a_l)])

svr_best = None
for C in [5, 10, 20, 40]:
    for g in [0.0001, 0.0002, 0.0003, 0.0005]:
        s = rmse(make_pipeline(RobustScaler(), SVR(C=C, epsilon=0.008, gamma=g)), X, y, kf)
        if svr_best is None or s < svr_best[0]: svr_best = (s, C, g)
print("best SVR", svr_best)

best Lasso alpha 0.0005 0.11175689374972449
best SVR (np.float64(0.1104189915963458), 20, 0.0005)


In [15]:
kr_best = None
for a in [0.3, 0.6, 1.0]:
    for g in [0.00005, 0.0001, 0.0002]:
        s = rmse(make_pipeline(RobustScaler(), KernelRidge(alpha=a, kernel="rbf", gamma=g)), X, y, kf)
        if kr_best is None or s < kr_best[0]: kr_best = (s, a, g)
print("best KernelRidge", kr_best)

best KernelRidge (np.float64(0.15251817993519462), 0.3, 0.0002)


In [16]:
models = {
    "lasso": make_pipeline(RobustScaler(), Lasso(alpha=a_l, max_iter=50000)),
    "enet": make_pipeline(RobustScaler(), ElasticNet(alpha=a_l, l1_ratio=0.9, max_iter=50000)),
    "svr": make_pipeline(RobustScaler(), SVR(C=svr_best[1], epsilon=0.008, gamma=svr_best[2])),
    "kr": make_pipeline(RobustScaler(), KernelRidge(alpha=kr_best[1], kernel="rbf", gamma=kr_best[2])),
    "gbr": GradientBoostingRegressor(n_estimators=3000, learning_rate=0.02, max_depth=4,
            max_features="sqrt", min_samples_leaf=15, min_samples_split=10,
            loss="huber", subsample=0.8, random_state=SEED),
    "hgb": HistGradientBoostingRegressor(learning_rate=0.03, max_iter=900, max_depth=4,
            min_samples_leaf=15, l2_regularization=1.0, random_state=SEED),
}

In [17]:
# optional libraries: used automatically if installed
try:
    from xgboost import XGBRegressor
    models["xgb"] = XGBRegressor(n_estimators=2500, learning_rate=0.01, max_depth=3,
        subsample=0.6, colsample_bytree=0.4, min_child_weight=2, reg_alpha=0.001,
        reg_lambda=0.8, random_state=SEED, n_jobs=-1)
except ImportError: pass
try:
    from lightgbm import LGBMRegressor
    models["lgbm"] = LGBMRegressor(objective="regression", num_leaves=6, learning_rate=0.01,
        n_estimators=4000, max_bin=200, bagging_fraction=0.75, bagging_freq=5,
        feature_fraction=0.25, min_data_in_leaf=6, random_state=SEED, verbose=-1)
except ImportError: pass
try:
    from catboost import CatBoostRegressor
    models["cat"] = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=5,
        l2_leaf_reg=3, loss_function="RMSE", random_seed=SEED, verbose=0)
except ImportError: pass
print("models:", list(models))

models: ['lasso', 'enet', 'svr', 'kr', 'gbr', 'hgb', 'xgb', 'lgbm']


In [18]:
# ---- 8. Stacking: out-of-fold predictions -> non-negative linear blender ----
from sklearn.linear_model import LinearRegression
kf_o = KFold(5, shuffle=True, random_state=SEED)
oof = np.zeros((len(y), len(models)))
for j, (name, m) in enumerate(models.items()):
    for tr, va in kf_o.split(X):
        oof[va, j] = clone(m).fit(X[tr], y[tr]).predict(X[va])
    print(f"{name:8s} OOF RMSLE: {np.sqrt(np.mean((oof[:, j]-y)**2)):.5f}")

blender = LinearRegression(positive=True, fit_intercept=False).fit(oof, y)
w = blender.coef_ / blender.coef_.sum()
print("blend weights:", dict(zip(models, np.round(w, 3))))
print(f"BLEND OOF RMSLE: {np.sqrt(np.mean((oof @ w - y)**2)):.5f}")

lasso    OOF RMSLE: 0.11198
enet     OOF RMSLE: 0.11210
svr      OOF RMSLE: 0.11075
kr       OOF RMSLE: 0.15269
gbr      OOF RMSLE: 0.11264
hgb      OOF RMSLE: 0.12016
xgb      OOF RMSLE: 0.11195
lgbm     OOF RMSLE: 0.11346
blend weights: {'lasso': np.float64(0.187), 'enet': np.float64(0.0), 'svr': np.float64(0.393), 'kr': np.float64(0.0), 'gbr': np.float64(0.135), 'hgb': np.float64(0.0), 'xgb': np.float64(0.285), 'lgbm': np.float64(0.0)}
BLEND OOF RMSLE: 0.10630


In [21]:
##9. Fit on all data and write submission ----
test_pred = np.column_stack([clone(m).fit(X, y).predict(X_test) for m in models.values()])
final = np.expm1(test_pred @ w)
pd.DataFrame({"Id": test_ids, "SalePrice": final}).to_csv("submission_v2.csv", index=False)
print("Saved submission_v2.csv")

Saved submission_v2.csv
